In [ ]:
import pandas as pd
import joblib
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from Preprocessing_pipeline import normalize_english

# 1. Download all required NLTK resources to avoid LookupError
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

# 2. Setup preprocessing parameters
english_StopWords = set(stopwords.words('english'))
punctuations = set(string.punctuation)
lemmatizer = WordNetLemmatizer()

# 3. Load & prepare dataset
df = pd.read_csv('../data/imdb_reviews.csv')
df = df.head(1000).copy()
df = df.dropna(subset=['review', 'sentiment']).copy()

df['clean_text'] = df['review'].apply(
    normalize_english,
    lemmatizer=lemmatizer,
    punctuations=punctuations,
    english_StopWords=english_StopWords
)

df['sentiment'] = df['sentiment'].astype(str).str.strip().str.capitalize()

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42
)

# 5. Build Pipeline Model
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=20000)),
    ('clf', LogisticRegression(C=2.0, max_iter=1000))
])

model.fit(X_train, y_train)

# 6. Evaluate & Save Model
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

joblib.dump(model, 'English_model_weights.pkl')
print("Model saved to English_model_weights.pkl")